In [ ]:
from PosSelect_Functions_Old import *
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import pandas as pd
import numpy as np
import copy
import seaborn as sns
from scipy.stats import mannwhitneyu as mwu
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import wilcoxon
from scipy.optimize import curve_fit
from scipy.stats import fisher_exact
from scipy.stats import norm
import os
from collections import Counter
from scipy.stats import binomtest

hfont = {'fontname':'Arial'}
plt.rcParams["font.family"] = "Arial"

#Code borrowed heavily from here: https://stackoverflow.com/questions/62375034/find-non-overlapping-area-between-two-kde-plots
plt.rcParams.update(
    {"text.usetex": False}
)

def parse_table(i):
    return [int(x) for x in i.replace("[", "").replace("]", "").split(",")]

d_abrev = {"LiangSteinNeuron":"FC exc. neur.", "FetalChondrocytes":"F chond.", "SertoliMale":"FG sertoli", "preGC_IIaFemale":"FG preGC IIa",\
          "NeuralFemale":"FG neur.", "FetalGonadImmuneFemale":"FG immune", "VIP":"AC VIP inh. neur.", "LiangSteinProgenitor":"FC prog.",\
          "AdultHeartVentricularCardiomyocyte":"AH cardiomyo.", "AdultLoopOfHenle":"AK loop of henle", "FetalBrainNeurGlioblast_CB_VZ":"FCB glioblast",\
         "AdultProximalTubule":"AK prox. tub.", "FetalLeydigMale":"FG leydig", "SST":"AC SST inh neur.", "KosoyRoussosControlMicroglia":"AC microglia",\
         "FetalBrainFloorPlate":"FB fl. plate", "FetalArterialECs":"FH endoth.", "ASCT":"AC astro.", "FetalBrainCOP":"FB COP",\
         "AMY":"AA neur.", "PVALB":"AC PVALB inh neur.", "ITL23":"AC L2-3 IT neur.", "FetalBrainNeurCB_GNP_IPC_1":"FB inter. prog.", "FetalBrainNeurDAergic":"FB DA neur.",\
          "OGC":"AC Oligo.", "D1Pu":"AP D1 inh neur.", "FetalBrainNeurSerotonergic":"FB 5-HT neur.", "FetalBrainNeurDRG_2":"FS DRG neur.",\
          "FetalHeartPericytes":"FH peri.", "FetalHeartEndocardium":"FH endocard.", "FetalHeartCardiacFibroblasts":"FH fibro.", "FetalBrainNeurPurkinje_6":"FCB Purk. inh neur.",\
          "AdultHeartSmoothMuscle":"AH smooth musc.", "FetalBrainRoofPlate":"FB ro. plate"}

palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey", "Human":"#E31A1C", "Chimp":"#0058FF"}

def binom_plot(resh, resc, title):
    plt.figure(figsize=(6, 4), dpi=600)
    cih = resh.proportion_ci(confidence_level=0.95)
    cic = resc.proportion_ci(confidence_level=0.95)
    # Small pseudocount to avoid log2(0) or division by zero
    eps = 0
    
    # Compute log2 fold-change = log2(p / (1 - p))
    log2_resh = np.log2((resh.statistic + eps) / (1 - resh.statistic + eps))
    log2_resc = np.log2((resc.statistic + eps) / (1 - resc.statistic + eps))
    
    # Compute CI bounds in the same way
    log2_ci_resh = [
        np.log2((cih.low + eps) / (1 - cih.low + eps)),
        np.log2((cih.high + eps) / (1 - cih.high + eps))
    ]
    
    log2_ci_resc = [
        np.log2((cic.low + eps) / (1 - cic.low + eps)),
        np.log2((cic.high + eps) / (1 - cic.high + eps))
    ]
    df = pd.DataFrame({
        "Condition": ["HH/HC", "CC/CH"],
        "Log2FC": [log2_resh, log2_resc],
        "CI_Low": [log2_ci_resh[0], log2_ci_resc[0]],
        "CI_High": [log2_ci_resh[1], log2_ci_resc[1]],
    })
    
    # Compute error bars
    df["Err_Low"] = df["Log2FC"] - df["CI_Low"]
    df["Err_High"] = df["CI_High"] - df["Log2FC"]
    plt.figure(figsize=(6,5))

    x = np.arange(len(df))
    
    plt.bar(
        x,
        df["Log2FC"],
        color=[palette["Human"], palette["Chimp"]],
        yerr=[df["Err_Low"], df["Err_High"]],
        capsize=5, alpha = 1
    )
    
    plt.xticks(x, ["Top 50 human accelerated", "Top 50 chimp accelerated"], size = 13)
    plt.ylabel("Log$_{2}$ fold-change for accel. derived\nrelative to non-accel. derived", size = 13)
    plt.title(title, size = 14)
    plt.yticks(size = 13)
    
    plt.show()

In [ ]:
#v = pd.read_csv("Fixed_FetalBrainNeurPurkinje_6.txt.gz", sep = "\t")
vv = pd.read_csv("Poly_MAF0.25_FetalBrainNeurPurkinje_6.txt.gz", sep = "\t")


try:
    vv = add_unfold(vv)
    vv_ref = vv[vv["Human ref"] == vv["Chimp ref"]]
    vv_alt = vv[vv["Human alt"] == vv["Chimp ref"]]
    vv_ref["fixed logfc"] = -vv_ref["logfc"].astype(float)
    vv_alt["fixed logfc"] = vv_alt["logfc"].astype(float)
    vv = pd.concat([vv_ref, vv_alt])
except:
    pass

te_blacklist = pd.read_csv("BlacklistTE_Variants.txt", sep = "\t")
    
#v = v[~v["Position"].isin(te_blacklist["Position"])]
vv = vv[~vv["Position"].isin(te_blacklist["Position"])]



In [ ]:
v = pd.read_csv("../AccelEvol_ForPub/FetalBrainNeurPurkinje_6_AllSitesToDownload.txt.gz", sep = "\t")

In [ ]:
p_cut = 3

In [ ]:
vx = v[v["SpecSup447"] > 248]
vvx = vv[vv["SpecSup447"] > 250]

vx["Chrom"] = [x.split(":")[0] for x in vx["Position"]]
vx["Pos"] = [int(x.split(":")[1]) for x in vx["Position"]]
vvx["Chrom"] = [x.split(":")[0] for x in vvx["Position"]]
vvx["Pos"] = [int(x.split(":")[1]) for x in vvx["Position"]]

inter = np.intersect1d(v["Position"], vv["Position"])

vx = vx[~vx["Position"].isin(inter)]
vxh = vx[vx["Derived"] == "H"]
vxc = vx[vx["Derived"] == "C"]

vxhu = vxh[(vxh["PhyloP447"] > p_cut) & (vxh["logfc"] < -0.5)]
vxhd = vxh[(vxh["PhyloP447"] > p_cut) & (vxh["logfc"] > 0.5)]
vxcu = vxc[(vxc["PhyloP447"] > p_cut) & (vxc["logfc"] < -0.5)]
vxcd = vxc[(vxc["PhyloP447"] > p_cut) & (vxc["logfc"] > 0.5)]

#Go through and get the polymorphisms that are near the fixed, conserved sites with large effect substitutions
khu = []
for index, row in vxhu.iterrows():
    khu = khu + [row["Chrom"] + ":" + str(x) for x in range(row["Pos"] - 100, row["Pos"] + 100)]

khd = []
for index, row in vxhd.iterrows():
    khd = khd + [row["Chrom"] + ":" + str(x) for x in range(row["Pos"] - 100, row["Pos"] + 100)]

kcu = []
for index, row in vxcu.iterrows():
    kcu = kcu + [row["Chrom"] + ":" + str(x) for x in range(row["Pos"] - 100, row["Pos"] + 100)]

kcd = []
for index, row in vxcd.iterrows():
    kcd = kcd + [row["Chrom"] + ":" + str(x) for x in range(row["Pos"] - 100, row["Pos"] + 100)]



In [ ]:
#Filter to just those sites
vvxhu = vvx[vvx["Position"].isin(khu)]
vvxhd = vvx[vvx["Position"].isin(khd)]
vvxcu = vvx[vvx["Position"].isin(kcu)]
vvxcd = vvx[vvx["Position"].isin(kcd)]

In [ ]:
#Set PhyloP cutoff
p_cut_comp = 3

In [ ]:
#Print out results of testing for differences in the PhyloP distribution and number of polymorphisms
ct = "Purkinje"
bh = vvxhd[vvxhd["PhyloP447"] > p_cut_comp].shape[0]
bc = vvxcd[vvxcd["PhyloP447"] > p_cut_comp].shape[0]
sh = vvxhd[vvxhd["PhyloP447"] < p_cut_comp].shape[0]
sc = vvxcd[vvxcd["PhyloP447"] < p_cut_comp].shape[0]
fe = fisher_exact([[bh, bc], [sh, sc]])
print([ct, "PhyloP", 0.5, "Near down", bh, bc, sh, sc, fe[0], fe[1]])

bh = vvxhu[vvxhu["PhyloP447"] > p_cut_comp].shape[0]
bc = vvxcu[vvxcu["PhyloP447"] > p_cut_comp].shape[0]
sh = vvxhu[vvxhu["PhyloP447"] < p_cut_comp].shape[0]
sc = vvxcu[vvxcu["PhyloP447"] < p_cut_comp].shape[0]
fe = fisher_exact([[bh, bc], [sh, sc]])
print([ct, "PhyloP", 0.5, "Near up", bh, bc, sh, sc, fe[0], fe[1]])

fh = vxhd.shape[0]
fc = vxcd.shape[0]
ph = vvxhd.shape[0]
pc = vvxcd.shape[0]
fe = fisher_exact([[ph, pc], [fh, fc]])
print([ct, "Total SNPs", 0.5, "Near down", ph, pc, fh, fc, fe[0], fe[1]])

fh = vxhu.shape[0]
fc = vxcu.shape[0]
ph = vvxhu.shape[0]
pc = vvxcu.shape[0]
fe = fisher_exact([[ph, pc], [fh, fc]])
print([ct, "Total SNPs", 0.5, "Near up", ph, pc, fh, fc, fe[0], fe[1]])

In [ ]:
sns.set_style("white")

In [ ]:
fig, ax = plt.subplots(figsize=(3, 4), dpi=600)
dfpp = pd.DataFrame([["Human-derived", 292], ["Chimp-derived", 263]])
dfpp.columns = ["Effect on CA", "Number of sites"]
sns.barplot(data = dfpp, x = "Effect on CA", y = "Number of sites", hue = "Effect on CA", alpha = 0.85, palette = {"Human-derived":"#E31A1C", "Chimp-derived":"#0058FF"})
plt.axhline(1662/(1662 + 1365)*(292 + 263), linestyle = "--", color = "black", xmin = 0.055, xmax = 0.4535)
plt.axhline(1365/(1662 + 1365)*(292 + 263), linestyle = "--", color = "black", xmin = 0.5 + 0.055, xmax = 0.5 + 0.4535)
plt.ylabel("Number of SNPs", size = 11)
plt.xlabel("Lineage", size = 11)
plt.xticks(size = 10)
plt.yticks(size = 10)
x1, x22 = 0, 1                 # bar positions
#y, h = 83, 0 
y, h = 330, 0 # height of line + vertical offset
plt.plot([x1, x1, x22, x22],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x22) * 0.5, y - 1,
         "NS",
         ha='center', va='bottom', fontsize=14)
plt.title("Subs. that decrease CA", size = 12)
plt.ylim(0, 360)
#plt.title("Purkinje cells, ATAC peaks near Tenm3", size = 15)

In [ ]:
ct = "Purkinje"
bd = vvxhd[vvxhd["PhyloP447"] > p_cut_comp].shape[0]
bu = vvxhu[vvxhu["PhyloP447"] > p_cut_comp].shape[0]
sd = vvxhd[vvxhd["PhyloP447"] < p_cut_comp].shape[0]
su = vvxhu[vvxhu["PhyloP447"] < p_cut_comp].shape[0]
fe = fisher_exact([[bd, bu], [sd, su]])
print([ct, "PhyloP", 0.5, "Near down", bd, bu, sd, su, fe[0], fe[1]])

fod = vxhd.shape[0]
fou = vxhu.shape[0]
pod = vvxhd.shape[0]
pou = vvxhu.shape[0]
fe = fisher_exact([[pod, pou], [fd, fu]])
print([ct, "Total SNPs", 0.5, "Near down", pod, pou, fd, fu, fe[0], fe[1]])

In [ ]:
fig, ax = plt.subplots(figsize=(3, 4), dpi=600)
dfpp = pd.DataFrame([["Decrease CA", 292], ["Increase CA", 122]])
dfpp.columns = ["Effect on CA", "Number of sites"]
sns.barplot(data = dfpp, x = "Effect on CA", y = "Number of sites", hue = "Effect on CA", alpha = 0.85, palette = {"Decrease CA":"magenta", "Increase CA":"green"})
plt.axhline(384/(384 + 119)*(292 + 122), linestyle = "--", color = "black", xmin = 0.055, xmax = 0.4535)
plt.axhline(119/(384 + 119)*(292 + 122), linestyle = "--", color = "black", xmin = 0.5 + 0.055, xmax = 0.5 + 0.4535)
plt.ylabel("Number of SNPs", size = 11)
plt.xlabel("Effect on CA", size = 11)
plt.xticks(size = 10)
plt.yticks(size = 10)
x1, x22 = 0, 1                 # bar positions
#y, h = 83, 0 
y, h = 330, 0 # height of line + vertical offset
plt.plot([x1, x1, x22, x22],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x22) * 0.5, y - 10,
         "*",
         ha='center', va='bottom', fontsize=14)
plt.title("Human-derived subs.", size = 12)
plt.ylim(0, 350)
#plt.title("Purkinje cells, ATAC peaks near Tenm3", size = 15)

In [ ]:
p_cut_comp = 6

ct = "Purkinje"
bh = vxhd[vxhd["PhyloP447"] > p_cut_comp].shape[0]
bc = vxcd[vxcd["PhyloP447"] > p_cut_comp].shape[0]
sh = vxhd[vxhd["PhyloP447"] < p_cut_comp].shape[0]
sc = vxcd[vxcd["PhyloP447"] < p_cut_comp].shape[0]
fe = fisher_exact([[bh, bc], [sh, sc]])
print([ct, "PhyloP", 0.5, "Near down", bh, bc, sh, sc, fe[0], fe[1]])

bh = vxhu[vxhu["PhyloP447"] > p_cut_comp].shape[0]
bc = vxcu[vxcu["PhyloP447"] > p_cut_comp].shape[0]
sh = vxhu[vxhu["PhyloP447"] < p_cut_comp].shape[0]
sc = vxcu[vxcu["PhyloP447"] < p_cut_comp].shape[0]
fe = fisher_exact([[bh, bc], [sh, sc]])
print([ct, "PhyloP", 0.5, "Near up", bh, bc, sh, sc, fe[0], fe[1]])

In [ ]:
print(ttest_ind(vxhu["PhyloP447"], vxcu["PhyloP447"]))
print(ttest_ind(np.maximum(vvxhu["PhyloP447"], 0), np.maximum(vvxcu["PhyloP447"], 0)))
print(ttest_ind(vxhd["PhyloP447"], vxcd["PhyloP447"]))
print(ttest_ind(np.maximum(vvxhd["PhyloP447"], 0), np.maximum(vvxcd["PhyloP447"], 0)))
print(ttest_ind(np.maximum(vvxhd["PhyloP447"], 0), np.maximum(vvxhu["PhyloP447"], 0)))
print(ttest_ind(np.maximum(vxhd["PhyloP447"], 0), np.maximum(vxhu["PhyloP447"], 0)))


In [ ]:
fig, ax = plt.subplots(figsize = (6, 4), dpi = 600)
sns.kdeplot({"Chimp-derived":np.abs(vxcd["PhyloP447"]), "Human-derived":np.abs(vxhd["PhyloP447"])}, common_norm = False, fill = True, palette = {"Human-derived":"#E31A1C", "Chimp-derived":"#0058FF"})
plt.title("Fixed subs. in conserved sites that decrease CA", size = 14)
plt.xlabel("PhyloP", size = 12)
plt.ylabel("Density", size = 12)
plt.xticks(size = 11)
plt.yticks(size = 11)

In [ ]:
fig, ax = plt.subplots(figsize = (6, 4), dpi = 600)
sns.kdeplot({"Decreasing CA":np.abs(vxhd["PhyloP447"]), "Increasing CA":np.abs(vxhu["PhyloP447"])}, common_norm = False, fill = True, palette = {"Decreasing CA":"magenta", "Increasing CA":"green"})
plt.title("Human-derived subs. in conserved sites", size = 14)
plt.xlabel("PhyloP", size = 12)
plt.ylabel("Density", size = 12)
plt.xticks(size = 11)
plt.yticks(size = 11)

In [ ]:
fig, ax = plt.subplots(figsize = (6, 4), dpi = 600)

sns.kdeplot({"Near chimp-derived":np.abs(vvxcd["PhyloP447"]), "Near human-derived":np.abs(vvxhd["PhyloP447"])}, common_norm = False, fill = True, palette = {"Near human-derived":"#E31A1C", "Near chimp-derived":"#0058FF"})
plt.title("Human SNPs near decreases in CA", size = 14)
plt.xlabel("PhyloP", size = 12)
plt.ylabel("Density", size = 12)
plt.xticks(size = 11)
plt.yticks(size = 11)

In [ ]:
fig, ax = plt.subplots(figsize = (6, 4), dpi = 600)

sns.kdeplot({"Near decrease in CA":np.abs(vvxhd["PhyloP447"]), "Near increase in CA":np.abs(vvxhu["PhyloP447"])}, common_norm = False, fill = True, palette = {"Near decrease in CA":"magenta", "Near increase in CA":"green"})
plt.title("SNPs near human-derived subs. in conserved sites", size = 14)
plt.xlabel("PhyloP", size = 12)
plt.ylabel("Density", size = 12)
plt.xticks(size = 11)
plt.yticks(size = 11)